# Final Project - One-Minute YOLO Epochs and 90% Precision Mode

This is the speed-focused copy of `01C_YOLO_BDD100K_Fast_High_Accuracy.ipynb`.
It leaves the executed `01C` notebook and all `v3` checkpoints unchanged. The default
profile was measured on this laptop at **29.9 seconds training**, **4.2 seconds epoch
validation**, and **46.9 seconds total first-run wall time**.

The notebook always compares the fast-tuned checkpoint against the untouched starting
checkpoint on full validation data. If the micro-training result is worse, deployment
keeps the starting checkpoint.

## What 90% means here

At confidence 0.70, the current checkpoint measured **90.5% aggregate precision** on the
1,200-image validation sample, with only **7.5% recall**. That is a valid high-precision
operating point, not 90% mAP or F1. The standard detector metrics remain much lower.

This notebook reports both without mixing them:

- **Standard quality:** precision, recall, F1, mAP@50, and mAP@50:95 at normal evaluation.
- **Strict deployment precision:** the validation operating point used for live detection.

A one-minute epoch contains 300 selected images, not all 70,000 BDD100K training images.
Full 8,000-image validation and 2,000-image test evaluation happen once after training and
will take longer than one minute.

In [ ]:
from pathlib import Path
import json
import random
import shutil
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_detection.yolo_training_utils import (
    latest_best_checkpoint,
    load_dataset_config,
    prepare_fast_data_files,
)

SEED = 74
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_float32_matmul_precision("high")

print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} | CUDA build: {torch.version.cuda}")

## Measured profile

The defaults below are the benchmarked settings. Raising `TRAIN_IMAGES`, `TRAIN_IMGSZ`,
or lowering `FREEZE_LAYERS` can improve learning capacity, but breaks the one-minute
constraint on this GPU.

In [ ]:
RUN_TAG = "v4_one_minute"
RUN_TRAINING = True
RESUME_TRAINING = False
START_WEIGHTS_OVERRIDE = None  # Example: r"runs/.../weights/best.pt"

EPOCHS = 60
TRAIN_IMAGES = 300
EPOCH_VAL_IMAGES = 100
TRAIN_IMGSZ = 320
TRAIN_BATCH = 32
FREEZE_LAYERS = 16
WORKERS = 0
DEVICE = 0
MAX_EPOCH_SECONDS = 60.0

EVAL_IMGSZ = 640
EVAL_BATCH = 16
TARGET_STANDARD_F1 = 0.90
TARGET_OPERATING_PRECISION = 0.90
MIN_OPERATING_RECALL = 0.01
HIGH_PRECISION_SCORE_FLOOR = 0.70
BALANCED_SCORE_FLOOR = 0.25

RUN_FULL_EVALUATION = True
RUN_TEST_EVALUATION = True
EXPORT_ONNX = False
EXPORT_TENSORRT = False

RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "yolo_accuracy"
DATA_YAML = PROJECT_ROOT / "data" / "bdd100k_yolo" / "data.yaml"
RUN_NAME = f"bdd100k_one_minute_{RUN_TAG}"
RUN_DIR = RUN_ROOT / RUN_NAME
MANIFEST_ROOT = RUN_ROOT / "manifests" / RUN_TAG
BASELINE_WEIGHTS = (
    PROJECT_ROOT / "runs" / "notebooks" / "yolo"
    / "bdd100k_cpu_quick_finetuned" / "weights" / "best.pt"
)

if START_WEIGHTS_OVERRIDE:
    START_WEIGHTS = Path(START_WEIGHTS_OVERRIDE)
    if not START_WEIGHTS.is_absolute():
        START_WEIGHTS = PROJECT_ROOT / START_WEIGHTS
else:
    try:
        START_WEIGHTS = latest_best_checkpoint(
            RUN_ROOT, fallback=BASELINE_WEIGHTS, exclude_run_names=(RUN_NAME,)
        )
    except FileNotFoundError:
        START_WEIGHTS = "yolo11n.pt"
        print("No local BDD100K checkpoint found; starting from COCO YOLO11n.")

preferred_tags = ("v3_fast", "v2")
PREFERRED_MANIFEST_TAG = next(
    (
        tag for tag in preferred_tags
        if (RUN_ROOT / "manifests" / tag / "refine.txt").exists()
    ),
    preferred_tags[0],
)
print(f"Starting checkpoint: {START_WEIGHTS}")
print(f"Preferred balanced manifest: {PREFERRED_MANIFEST_TAG}")

## Verify CUDA

In [ ]:
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Start Jupyter with C:\tf214_hw2 and the CUDA PyTorch wheel."
    )
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GB")
    x = torch.randn(1024, 1024, device="cuda")
    _ = x @ x
    torch.cuda.synchronize()
    del x
    torch.cuda.empty_cache()

## Build the 300-image rare-class micro-epoch

The training set comes from the strongest balanced refinement manifest available. It is
cached in RAM and uses only 100 fixed images for epoch-to-epoch feedback.

In [ ]:
assert DATA_YAML.exists(), f"Missing converted dataset: {DATA_YAML}"
data_config, dataset_root, class_names = load_dataset_config(DATA_YAML)
fast_data = prepare_fast_data_files(
    data_yaml=DATA_YAML,
    output_dir=MANIFEST_ROOT,
    run_root=RUN_ROOT,
    preferred_manifest_tag=PREFERRED_MANIFEST_TAG,
    main_count=TRAIN_IMAGES,
    refine_count=TRAIN_IMAGES,
    validation_count=EPOCH_VAL_IMAGES,
    seed=SEED,
)
MICRO_DATA_YAML = fast_data.refine_yaml

manifest_summary = pd.DataFrame([
    {"purpose": "micro-epoch training", "images": sum(1 for _ in fast_data.refine_manifest.open(encoding="utf-8"))},
    {"purpose": "per-epoch validation", "images": sum(1 for _ in fast_data.validation_manifest.open(encoding="utf-8"))},
])
print(f"Manifest source: {fast_data.source}")
display(manifest_summary)

## Starting checkpoint evidence

These are the two epochs from the preserved `v3` run. They took 25.0 and 31.2 minutes,
which is why this notebook changes the definition and size of an epoch.

In [ ]:
start_path = Path(START_WEIGHTS)
prior_results = start_path.parent.parent / "results.csv" if start_path.exists() else None
if prior_results is not None and prior_results.exists():
    prior_history = pd.read_csv(prior_results)
    epoch_seconds = prior_history["time"].diff().fillna(prior_history["time"])
    prior_table = prior_history[[
        "epoch", "metrics/precision(B)", "metrics/recall(B)",
        "metrics/mAP50(B)", "metrics/mAP50-95(B)"
    ]].copy()
    prior_table["epoch_minutes"] = epoch_seconds / 60.0
    display(prior_table.tail().style.format({
        "metrics/precision(B)": "{:.3f}",
        "metrics/recall(B)": "{:.3f}",
        "metrics/mAP50(B)": "{:.3f}",
        "metrics/mAP50-95(B)": "{:.3f}",
        "epoch_minutes": "{:.1f}",
    }))
else:
    print("No prior training history was found.")

## Train one-minute micro-epochs

Only the upper neck and detection head remain trainable. Mild color and scale augmentation
provides variation without the CPU cost of mosaic. The low learning rate limits damage to
the already-adapted BDD100K features.

In [ ]:
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
if RUN_TRAINING:
    if RESUME_TRAINING:
        assert LAST_WEIGHTS.exists(), f"No resumable checkpoint: {LAST_WEIGHTS}"
        train_result = YOLO(str(LAST_WEIGHTS)).train(resume=True)
    else:
        if RUN_DIR.exists():
            raise FileExistsError(
                f"{RUN_DIR} already exists. Set RESUME_TRAINING=True or change RUN_TAG."
            )
        train_result = YOLO(str(START_WEIGHTS)).train(
            data=str(MICRO_DATA_YAML),
            epochs=EPOCHS,
            imgsz=TRAIN_IMGSZ,
            batch=TRAIN_BATCH,
            device=DEVICE,
            workers=WORKERS,
            cache="ram",
            project=str(RUN_ROOT),
            name=RUN_NAME,
            exist_ok=False,
            pretrained=True,
            optimizer="AdamW",
            lr0=1e-4,
            lrf=0.10,
            weight_decay=7e-4,
            patience=12,
            cos_lr=True,
            warmup_epochs=1.0,
            freeze=FREEZE_LAYERS,
            box=7.5,
            cls=0.70,
            cls_pw=0.80,
            mosaic=0.0,
            mixup=0.0,
            hsv_h=0.010,
            hsv_s=0.25,
            hsv_v=0.30,
            translate=0.04,
            scale=0.20,
            fliplr=0.50,
            close_mosaic=0,
            amp=True,
            channels_last=True,
            deterministic=False,
            seed=SEED,
            max_det=100,
            plots=False,
            save=True,
            save_period=5,
            val=True,
            verbose=True,
        )

MICRO_BEST = RUN_DIR / "weights" / "best.pt"
assert MICRO_BEST.exists(), f"Micro-training checkpoint not found: {MICRO_BEST}"
MICRO_BEST

## Verify measured epoch time

In [ ]:
history = pd.read_csv(RUN_DIR / "results.csv")
history["epoch_seconds"] = history["time"].diff().fillna(history["time"])
history["under_one_minute"] = history["epoch_seconds"] <= MAX_EPOCH_SECONDS
columns = [
    "epoch", "epoch_seconds", "under_one_minute",
    "metrics/precision(B)", "metrics/recall(B)",
    "metrics/mAP50(B)", "metrics/mAP50-95(B)",
]
display(history[columns].tail(20).style.format({
    "epoch_seconds": "{:.1f}",
    "metrics/precision(B)": "{:.3f}",
    "metrics/recall(B)": "{:.3f}",
    "metrics/mAP50(B)": "{:.3f}",
    "metrics/mAP50-95(B)": "{:.3f}",
}))
print(f"Median epoch time: {history['epoch_seconds'].median():.1f} seconds")
print(f"Sub-minute epochs: {int(history['under_one_minute'].sum())}/{len(history)}")

## Full no-regression model selection

This is intentionally slower than one minute because both models are evaluated on all
8,000 validation images at 640px. It runs once, after the fast training loop.

In [ ]:
def summarize_metrics(label, metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    return {
        "candidate": label,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": map50,
        "mAP50_95": map50_95,
        "selection_score": 0.65 * f1 + 0.35 * map50,
    }

if not RUN_FULL_EVALUATION:
    raise RuntimeError("RUN_FULL_EVALUATION must remain True for honest model selection.")

candidate_paths = {"starting": START_WEIGHTS, "micro_tuned": MICRO_BEST}
full_val_metrics = {}
full_rows = []
for label, weights in candidate_paths.items():
    metrics = YOLO(str(weights)).val(
        data=str(DATA_YAML), split="val",
        imgsz=EVAL_IMGSZ, batch=EVAL_BATCH,
        device=DEVICE, workers=WORKERS,
        conf=0.001, iou=0.70, max_det=300,
        plots=False, verbose=False,
    )
    full_val_metrics[label] = metrics
    full_rows.append(summarize_metrics(label, metrics))

candidate_table = pd.DataFrame(full_rows).sort_values("selection_score", ascending=False)
display(candidate_table.style.format({
    column: "{:.3f}" for column in candidate_table.columns if column != "candidate"
}))
SELECTED_LABEL = str(candidate_table.iloc[0]["candidate"])
BEST_WEIGHTS = candidate_paths[SELECTED_LABEL]
val_metrics = full_val_metrics[SELECTED_LABEL]
best_model = YOLO(str(BEST_WEIGHTS))
print(f"Selected checkpoint: {SELECTED_LABEL} -> {BEST_WEIGHTS}")

## Held-out test metrics

In [ ]:
if RUN_TEST_EVALUATION:
    test_metrics = best_model.val(
        data=str(DATA_YAML), split="test",
        imgsz=EVAL_IMGSZ, batch=EVAL_BATCH,
        device=DEVICE, workers=WORKERS,
        conf=0.001, iou=0.70, max_det=300,
        plots=False, verbose=False,
    )
else:
    test_metrics = None

summary_rows = [{**summarize_metrics("validation", val_metrics), "split": "validation"}]
if test_metrics is not None:
    summary_rows.append({**summarize_metrics("test", test_metrics), "split": "test"})
summary = pd.DataFrame(summary_rows).drop(columns=["candidate", "selection_score"])
display(summary.style.format({
    column: "{:.3f}" for column in summary.columns if column != "split"
}))

## Measure the strict 0.70 operating point from full-validation curves

This uses the full-validation prediction curves already computed above, so it does not
need another validation pass.

In [ ]:
def operating_point(metrics, threshold):
    px = np.asarray(metrics.box.px)
    p_curve = np.asarray(metrics.box.p_curve)
    r_curve = np.asarray(metrics.box.r_curve)
    class_precision = np.array([
        np.interp(threshold, px, curve) for curve in p_curve
    ])
    class_recall = np.array([
        np.interp(threshold, px, curve) for curve in r_curve
    ])
    precision = float(np.nanmean(class_precision))
    recall = float(np.nanmean(class_recall))
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {
        "confidence": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

strict_operating_point = operating_point(val_metrics, HIGH_PRECISION_SCORE_FLOOR)
display(pd.DataFrame([strict_operating_point]).style.format("{:.3f}"))

## Calibrate balanced and 90%-precision class thresholds

Each class is handled separately. Classes that cannot reach 90% precision while keeping
at least 1% recall are marked as fallbacks rather than silently reported as successful.

In [ ]:
def calibrate_profile(metrics, name, score_floor, precision_target=None):
    px = np.asarray(metrics.box.px)
    f1_curve = np.asarray(metrics.box.f1_curve)
    p_curve = np.asarray(metrics.box.p_curve)
    r_curve = np.asarray(metrics.box.r_curve)
    rows = []
    thresholds = {}

    for position, class_id_raw in enumerate(metrics.ap_class_index):
        class_id = int(class_id_raw)
        best_f1_index = int(np.nanargmax(f1_curve[position]))
        selected_index = best_f1_index
        selection = "best F1"
        target_met = precision_target is None
        if precision_target is not None:
            valid = np.flatnonzero(
                (p_curve[position] >= precision_target)
                & (r_curve[position] >= MIN_OPERATING_RECALL)
            )
            if len(valid):
                selected_index = int(valid[np.nanargmax(r_curve[position, valid])])
                selection = "90% precision target"
                target_met = True
            else:
                selection = "target unavailable; score-floor fallback"

        threshold = max(score_floor, float(px[selected_index]))
        thresholds[str(class_id)] = threshold
        estimated_precision = float(np.interp(threshold, px, p_curve[position]))
        estimated_recall = float(np.interp(threshold, px, r_curve[position]))
        if precision_target is not None:
            target_met = target_met and estimated_precision >= precision_target
        rows.append({
            "profile": name,
            "class_id": class_id,
            "class": best_model.names[class_id],
            "threshold": threshold,
            "estimated_precision": estimated_precision,
            "estimated_recall": estimated_recall,
            "best_f1": float(f1_curve[position, best_f1_index]),
            "target_met": target_met,
            "selection": selection,
        })
    return thresholds, pd.DataFrame(rows)

balanced_thresholds, balanced_calibration = calibrate_profile(
    val_metrics, "balanced", BALANCED_SCORE_FLOOR
)
precision90_thresholds, precision90_calibration = calibrate_profile(
    val_metrics, "high_precision_90", HIGH_PRECISION_SCORE_FLOOR,
    precision_target=TARGET_OPERATING_PRECISION,
)
calibration_table = pd.concat(
    [balanced_calibration, precision90_calibration], ignore_index=True
)
display(calibration_table.style.format({
    "threshold": "{:.3f}",
    "estimated_precision": "{:.3f}",
    "estimated_recall": "{:.3f}",
    "best_f1": "{:.3f}",
}))

## Save the selected deployment model

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "yolo_one_minute"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEPLOY_WEIGHTS = OUTPUT_ROOT / "bdd100k_yolo_selected.pt"
DEPLOY_CONFIG = OUTPUT_ROOT / "deployment_config.json"
shutil.copy2(BEST_WEIGHTS, DEPLOY_WEIGHTS)

threshold_profiles = {
    "balanced": {
        "inference_confidence": min(balanced_thresholds.values()),
        "class_thresholds": balanced_thresholds,
    },
    "high_precision_90": {
        "inference_confidence": min(precision90_thresholds.values()),
        "class_thresholds": precision90_thresholds,
    },
}
deployment = {
    "model_family": "YOLO11",
    "selected_stage": SELECTED_LABEL,
    "weights": str(DEPLOY_WEIGHTS.resolve()),
    "imgsz": EVAL_IMGSZ,
    "max_det": 100,
    "default_threshold_profile": "high_precision_90",
    "threshold_profiles": threshold_profiles,
    "inference_confidence": threshold_profiles["high_precision_90"]["inference_confidence"],
    "class_thresholds": precision90_thresholds,
    "class_names": {str(key): value for key, value in best_model.names.items()},
    "target_operating_precision": TARGET_OPERATING_PRECISION,
    "minimum_display_confidence": HIGH_PRECISION_SCORE_FLOOR,
    "standard_validation_metrics": summarize_metrics("validation", val_metrics),
    "strict_operating_point": strict_operating_point,
}
DEPLOY_CONFIG.write_text(json.dumps(deployment, indent=2), encoding="utf-8")
print(f"Weights: {DEPLOY_WEIGHTS}")
print(f"Configuration: {DEPLOY_CONFIG}")
print(
    "Live command:\n"
    f'python -m road_detection.realtime_detect --backend yolo '
    f'--weights "{DEPLOY_WEIGHTS}" --config "{DEPLOY_CONFIG}" '
    f'--threshold-profile high_precision_90 --source 0 --device 0'
)

## Fast FP16 inference check

In [ ]:
deployment_model = YOLO(str(DEPLOY_WEIGHTS))
test_images = sorted((dataset_root / "images" / "test").glob("*.jpg"))
benchmark_images = test_images[:min(30, len(test_images))]
benchmark_frames = [cv2.imread(str(path)) for path in benchmark_images]
benchmark_frames = [frame for frame in benchmark_frames if frame is not None]
inference_confidence = threshold_profiles["high_precision_90"]["inference_confidence"]
deployment_model.fuse()
for frame in benchmark_frames[:3]:
    deployment_model.predict(
        frame, imgsz=EVAL_IMGSZ, conf=inference_confidence,
        device=DEVICE, quantize=16, max_det=100, verbose=False,
    )
if torch.cuda.is_available():
    torch.cuda.synchronize()

inference_times = []
started = time.perf_counter()
for frame in benchmark_frames:
    result = deployment_model.predict(
        frame, imgsz=EVAL_IMGSZ, conf=inference_confidence,
        device=DEVICE, quantize=16, max_det=100, verbose=False,
    )[0]
    inference_times.append(float(result.speed["inference"]))
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - started
end_to_end_fps = len(benchmark_frames) / max(elapsed, 1e-9)
model_fps = 1000.0 / max(float(np.mean(inference_times)), 1e-9)
print(f"End-to-end speed: {end_to_end_fps:.2f} FPS")
print(f"Model inference only: {model_fps:.2f} FPS")

## Honest acceptance table

In [ ]:
validation_row = summary.loc[summary["split"] == "validation"].iloc[0]
precision90_rows = calibration_table[
    calibration_table["profile"] == "high_precision_90"
]
acceptance = pd.DataFrame([
    {
        "requirement": "median training epoch <= 60 seconds",
        "measured": float(history["epoch_seconds"].median()),
        "target": MAX_EPOCH_SECONDS,
        "passed": float(history["epoch_seconds"].median()) <= MAX_EPOCH_SECONDS,
    },
    {
        "requirement": "standard validation F1 >= 90%",
        "measured": float(validation_row["f1"]),
        "target": TARGET_STANDARD_F1,
        "passed": float(validation_row["f1"]) >= TARGET_STANDARD_F1,
    },
    {
        "requirement": "strict operating precision >= 90%",
        "measured": strict_operating_point["precision"],
        "target": TARGET_OPERATING_PRECISION,
        "passed": strict_operating_point["precision"] >= TARGET_OPERATING_PRECISION,
    },
    {
        "requirement": "classes meeting 90% precision target",
        "measured": int(precision90_rows["target_met"].sum()),
        "target": len(precision90_rows),
        "passed": bool(precision90_rows["target_met"].all()),
    },
])
display(acceptance.style.format({"measured": "{:.3f}", "target": "{:.3f}"}))

if strict_operating_point["precision"] >= TARGET_OPERATING_PRECISION:
    print(
        f"The 90% strict-precision target is met at confidence >= "
        f"{HIGH_PRECISION_SCORE_FLOOR:.2f}, with estimated recall "
        f"{strict_operating_point['recall']:.1%}."
    )
else:
    print("The measured 90% strict-precision target is not met.")
print(
    "Do not describe strict operating precision as mAP or overall accuracy. "
    "Use the standard validation/test rows for detector quality."
)

## Optional exports

In [ ]:
exported = {}
if EXPORT_ONNX:
    exported["onnx"] = deployment_model.export(
        format="onnx", imgsz=EVAL_IMGSZ, dynamic=False,
        simplify=True, half=False
    )
if EXPORT_TENSORRT:
    exported["engine"] = deployment_model.export(
        format="engine", imgsz=EVAL_IMGSZ, batch=1,
        dynamic=False, simplify=True, half=True, device=0
    )
exported